# 07 — VLM second-pass extraction

## 1. Установка зависимостей

In [ ]:
!pip install -q qwen-vl-utils

## 2. Загрузка модели

Qwen2.5-VL-7B-Instruct-AWQ — 4-bit квантизованная версия, ~7 GB VRAM.

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info


MODEL_ID = 'Qwen/Qwen2.5-VL-7B-Instruct'

print(f'CUDA: {torch.cuda.is_available()}, devices: {torch.cuda.device_count()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

print(f'Model loaded: {MODEL_ID}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        used = torch.cuda.memory_allocated(i) / 1e9
        print(f'  GPU {i} VRAM: {used:.1f} GB')

## 3. Поиск входных данных

На Kaggle датасеты лежат в `/kaggle/input/<dataset-name>/`. Найдём через glob, не привязываясь к конкретному имени.

In [ ]:
import json
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
OUTPUT = Path('/kaggle/working/vlm_cache_v2.json')

# Ищем best_crops_meta.json
meta_candidates = list(INPUT_ROOT.rglob('best_crops_meta.json'))
assert meta_candidates, 'best_crops_meta.json не найден в /kaggle/input/. Загрузи как Kaggle dataset.'
META_PATH = meta_candidates[0]
print(f'META: {META_PATH}')

# Ищем папку best_crops/ — берём корень первого найденного <vid>/0000_0.jpg
crop_samples = list(INPUT_ROOT.rglob('*_0.jpg'))[:5]
assert crop_samples, 'Не найдены файлы вида <tid>_<rank>.jpg в /kaggle/input/'
# Корень best_crops/ = родитель родителя файла (best_crops/<vid>/<tid>_<rank>.jpg)
CROPS_ROOT = crop_samples[0].parent.parent
print(f'CROPS ROOT: {CROPS_ROOT}')
print(f'Subdirs (videos): {[p.name for p in CROPS_ROOT.iterdir() if p.is_dir()][:10]}')

meta_all = json.loads(META_PATH.read_text(encoding='utf-8'))
print(f'\nTracks: {len(meta_all)}, total crops: {sum(len(t["crops"]) for t in meta_all)}')

## 4. Промпт и парсер JSON

In [ ]:
import re

VLM_PROMPT = '''Ты эксперт по распознаванию ценников супермаркета "Лента". На изображении — один ценник.

Извлеки СТРОГО следующие 4 поля. ЧИТАЙ ТОЛЬКО ТО ЧТО ФИЗИЧЕСКИ ВИДИШЬ НА КАРТИНКЕ.
Не подставляй типичные значения, не угадывай по контексту.

1. **print_datetime** — дата и время печати ценника. Формат: "DD.MM.YYYY H:MM" (день.месяц.год часы:минуты). Печатается МЕЛКИМ шрифтом в нижней части ценника, обычно одной строкой.

2. **id_sku** — артикул товара. Число из 9-12 цифр без пробелов. Печатается рядом с датой печати.

3. **barcode** — штрих-код EAN-13. РОВНО 13 цифр без пробелов. Печатается ПОД полосатым штрих-кодом, часто отдельной строкой.

4. **code** — внутренний код в формате "NNNNNN - NNNNNN" (два блока по 4-7 цифр через дефис). Печатается в нижней части ценника.

КРИТИЧЕСКИ ВАЖНЫЕ ПРАВИЛА:
- Если поле НЕ ВИДНО, размыто, обрезано, или ты не уверен — обязательно поставь `null`.
- `null` — это ПРАВИЛЬНЫЙ и ЧАСТЫЙ ответ. Большинство ценников имеют 1-2 нечитаемых поля.
- НЕ ПРИДУМЫВАЙ значения. Никогда не возвращай числа которые "обычно бывают" или "похоже на правду".
- Все цифры должны быть прочитаны посимвольно с этой конкретной картинки.
- Если ты видишь только ЧАСТЬ числа (например 11 из 13 цифр barcode) — поставь `null`, не дописывай.
- Возвращай ТОЛЬКО валидный JSON, без markdown, без комментариев, без объяснений.

Схема ответа:
{"print_datetime": <строка по формату DD.MM.YYYY H:MM или null>, "id_sku": <строка из 9-12 цифр или null>, "barcode": <строка из 13 цифр или null>, "code": <строка вида "NNNNNN - NNNNNN" или null>}'''


def parse_vlm_response(text):
    """Достаёт JSON из ответа модели. Возвращает dict или None."""
    matches = re.findall(r'\{[^{}]*\}', text, re.DOTALL)
    if not matches:
        return None
    for raw in sorted(matches, key=len, reverse=True):
        try:
            obj = json.loads(raw)
            if isinstance(obj, dict):
                return obj
        except Exception:
            continue
    return None


# Тест парсера
test = '''Вот результат:\n{"print_datetime": "27.01.2026 14:00", "id_sku": "270123794224", "barcode": null, "code": null}\n'''
print(parse_vlm_response(test))

## 5. Inference helper

In [ ]:
def vlm_extract(image_path: Path) -> dict:
    """Один кроп → dict с 4 полями (значения или None)."""
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': str(image_path)},
            {'type': 'text', 'text': VLM_PROMPT},
        ],
    }]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors='pt',
    ).to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=160,
            do_sample=False,
            temperature=None,
            top_p=None,
        )

    # Обрезаем prompt-токены
    output_trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, output_ids)]
    response = processor.batch_decode(output_trimmed, skip_special_tokens=True)[0]

    parsed = parse_vlm_response(response)
    if parsed is None:
        return {'print_datetime': None, 'id_sku': None, 'barcode': None, 'code': None}

    out = {}
    for k in ('print_datetime', 'id_sku', 'barcode', 'code'):
        v = parsed.get(k)
        if v in (None, '', 'null', 'нет'):
            out[k] = None
        else:
            out[k] = str(v).strip()
    return out


first_crop_rel = meta_all[0]['crops'][0]['crop_path']  # 'data/best_crops/<vid>/<tid>_<rank>.jpg'
vid = meta_all[0]['video']
tid_rank = Path(first_crop_rel).name  # '<tid>_<rank>.jpg'
first_path = CROPS_ROOT / vid / tid_rank
print(f'Smoke-test crop: {first_path}  exists={first_path.exists()}')
if first_path.exists():
    result = vlm_extract(first_path)
    print(f'Result: {result}')

## 6. Прогон по всем трекам с консенсусом

На каждый трек — top-3 самых резких кропа (rank 0, 1, 2). Majority vote по полю; при равенстве — берём результат с самого резкого кропа.

In [ ]:
from collections import Counter
import time

N_CROPS_PER_TRACK = 3   # consensus по top-3 (rank 0,1,2)
FIELDS = ('print_datetime', 'id_sku', 'barcode', 'code')


def consensus(crop_results: list[dict]) -> dict:
    """Majority vote per field. None считается как «не голосовал»."""
    out = {}
    for f in FIELDS:
        values = [r[f] for r in crop_results if r.get(f) is not None]
        if not values:
            out[f] = None
            continue
        c = Counter(values)
        top_v, top_n = c.most_common(1)[0]
        out[f] = top_v
    return out


vlm_cache = {}
t0 = time.time()

for i, track in enumerate(meta_all, 1):
    vid = track['video']
    tid = track['track_id']
    key = f'{vid}/{tid}'


    top_crops = track['crops'][:N_CROPS_PER_TRACK]

    crop_results = []
    for c in top_crops:
        crop_path = CROPS_ROOT / vid / Path(c['crop_path']).name
        if not crop_path.exists():
            continue
        try:
            r = vlm_extract(crop_path)
        except Exception as e:
            print(f'  ! {crop_path}: {e}')
            r = {f: None for f in FIELDS}
        crop_results.append(r)

    vlm_cache[key] = consensus(crop_results)

    if i % 25 == 0:
        rate = i / (time.time() - t0)
        eta_min = (len(meta_all) - i) / rate / 60
        # Сколько полей заполнено в среднем
        filled = sum(sum(1 for v in vlm_cache[f'{m["video"]}/{m["track_id"]}'].values() if v is not None)
                     for m in meta_all[:i] if f'{m["video"]}/{m["track_id"]}' in vlm_cache)
        print(f'  {i}/{len(meta_all)} ({rate*60:.1f} tr/min, ETA {eta_min:.1f} min, '
              f'avg filled fields: {filled/i:.2f}/4)')
        # Промежуточный сейв
        OUTPUT.write_text(json.dumps(vlm_cache, ensure_ascii=False, indent=1))

OUTPUT.write_text(json.dumps(vlm_cache, ensure_ascii=False, indent=1))
elapsed = (time.time() - t0) / 60
print(f'\nГотово за {elapsed:.1f} мин, треков: {len(vlm_cache)}')
print(f'Сохранено: {OUTPUT}')

## 7. Статистика заполнения

In [ ]:
print(f'{"FIELD":<20} {"FILLED":>10} {"% of tracks":>15}')
print('─' * 50)
for f in FIELDS:
    n = sum(1 for v in vlm_cache.values() if v.get(f) is not None)
    print(f'{f:<20} {n:>10} {n/len(vlm_cache):>14.1%}')

print(f'\nПримеры (первые 5):')
for i, (k, v) in enumerate(list(vlm_cache.items())[:5]):
    print(f'  {k}: {v}')